# Logging, evaluating, and tracing Cerebras models with Braintrust

## Setup

Let's install some dependencies.


In [ ]:
%pip install autoevals braintrust openai

To use Cerebras models, configure your Cerebras API key in Braintrust:

- Get a Cerebras API key from [Cerebras Cloud](https://cloud.cerebras.ai?utm_source=braintrust)
- Add the Cerebras API key to your organization's AI providers
- Set the Cerebras API key and your Braintrust API key as environment variables

In [ ]:
# In your .env file

CEREBRAS_API_KEY=<your-cerebras-api-key>
BRAINTRUST_API_KEY=<your-braintrust-api-key>
 
# If you are self-hosting Braintrust, set the URL of your hosted dataplane
# BRAINTRUST_API_URL=<your-braintrust-api-url>

Braintrust knows how to intercept calls to the `openai` client library to automatically trace them. Since Cerebras has an OpenAI-compatible API, it's a breeze to set this up!


In [ ]:
import os

import openai
import braintrust

client = braintrust.wrap_openai(
    openai.OpenAI(
        api_key=os.getenv("CEREBRAS_API_KEY"),
        base_url="https://api.cerebras.ai/v1",
    )
)

## Logging

To log to Braintrust, simply initialize a logger. All Cerebras model calls will be automatically traced and logged to Braintrust. This works for both streaming and non-streaming calls.


In [10]:
braintrust.init_logger("Cerebras test")

response = client.chat.completions.create(
    model="llama3.1-8b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of the Nevada?"},
    ],
)

print(response.choices[0].message.content)

The capital of Nevada is Carson City.


In Braintrust, we'll see the completion along with a bunch of metrics. Wow, Cerebras is fast!

![Log view](./assets/Log-view.png)



If you enter your Cerebras API key in Braintrust (under Settings -> AI providers), you can also reproduce the call in the UI, and even tweak the prompt!

![Tweak prompt](./assets/Tweak-prompt.gif)

## Evaluating

Evals automatically support Cerebras models as well. Let's run a simple math test eval and see how it does.


In [16]:
from braintrust import Eval
from autoevals import Factuality

Eval(
    "Cerebras test",
    data=[
        {"input": "What is 100-94?", "expected": "6"},
        {"input": "square root of 16?", "expected": "4"},
    ],
    task=lambda input: client.chat.completions.create(
        model="llama3.1-8b",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": input},
        ],
    )
    .choices[0]
    .message.content,
    # We'll use the smarter Llama 3.3-70b model to evaluate the output.
    scores=[Factuality(model="llama3.3-70b", api_key=os.environ["CEREBRAS_API_KEY"])],
)


Experiment main-1758911835 is running at https://www.braintrust.dev/app/kevinwt/p/Cerebras%20test/experiments/main-1758911835


<Task pending name='Task-12' coro=<_EvalCommon.<locals>.run_to_completion() running at /opt/homebrew/lib/python3.11/site-packages/braintrust/framework.py:771>>

Cerebras test (data): 2it [00:00, 6689.48it/s]
Cerebras test (tasks): 100%|██████████| 2/2 [00:01<00:00,  1.30it/s]



=========================SUMMARY=========================
main-1758911835 compared to main-1758911800:
60.00% 'Factuality' score

0.75tok (+61.07%) 'time_to_first_token'         	(0 improvements, 2 regressions)
2 (-) 'llm_calls'                   	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                  	(0 improvements, 0 regressions)
0 (-100.00%) 'errors'                      	(2 improvements, 0 regressions)
0 (-) 'llm_errors'                  	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                 	(0 improvements, 0 regressions)
47.50tok (-) 'prompt_tokens'               	(0 improvements, 0 regressions)
0tok (-) 'prompt_cached_tokens'        	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'	(0 improvements, 0 regressions)
25.50tok (+900.00%) 'completion_tokens'           	(0 improvements, 2 regressions)
73tok (+900.00%) 'total_tokens'                	(0 improvements, 2 regressions)
0.00$ (+00.00%) 'estimated_cost'              	(0 imp

Looks like the output is getting penalized for containing an explanation of how to solve the problem. Let's tweak the prompt and try again.


In [ ]:
from braintrust import Eval
from autoevals import Factuality

Eval(
    "Cerebras test",
    data=[
        {"input": "What is 100-94?", "expected": "6"},
        {"input": "square root of 16?", "expected": "4"},
    ],
    task=lambda input: client.chat.completions.create(
        model="llama3.1-8b",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Solve the problem and provide the answer only.",
            },
            {"role": "user", "content": input},
        ],
    )
    .choices[0]
    .message.content,
    # We'll use the smarter Llama 3.3-70b model to evaluate the output.
    scores=[Factuality(model="llama3.3-70b", api_key=os.environ["CEREBRAS_API_KEY"])],
)

Experiment main-1758911844 is running at https://www.braintrust.dev/app/kevinwt/p/Cerebras%20test/experiments/main-1758911844


<Task pending name='Task-19' coro=<_EvalCommon.<locals>.run_to_completion() running at /opt/homebrew/lib/python3.11/site-packages/braintrust/framework.py:771>>

Cerebras test (data): 2it [00:00, 20815.40it/s]
Cerebras test (tasks): 100%|██████████| 2/2 [00:00<00:00,  2.18it/s]



=========================SUMMARY=========================
main-1758911844 compared to main-1758911835:
100.00% (+40.00%) 'Factuality' score	(2 improvements, 0 regressions)

0.40tok (-35.52%) 'time_to_first_token'         	(2 improvements, 0 regressions)
2 (-) 'llm_calls'                   	(0 improvements, 0 regressions)
0 (-) 'tool_calls'                  	(0 improvements, 0 regressions)
0 (-) 'errors'                      	(0 improvements, 0 regressions)
0 (-) 'llm_errors'                  	(0 improvements, 0 regressions)
0 (-) 'tool_errors'                 	(0 improvements, 0 regressions)
56.50tok (+900.00%) 'prompt_tokens'               	(0 improvements, 2 regressions)
0tok (-) 'prompt_cached_tokens'        	(0 improvements, 0 regressions)
0tok (-) 'prompt_cache_creation_tokens'	(0 improvements, 0 regressions)
2tok (-2350.00%) 'completion_tokens'           	(2 improvements, 0 regressions)
58.50tok (-1450.00%) 'total_tokens'                	(2 improvements, 0 regressions)
0.00$ (-0

Excellent! It looks like we improved both cases.

![Updated eval](./assets/Eval-2.gif)

## Where to go from here

Now that you can build logs and evaluations for your Cerebras models, you can ship applications with the confidence that you can reproduce user issues,
eval to improve your prompts, and continue to iterate with confidence.

To learn more about Braintrust, check out the [docs](https://braintrust.dev/docs).
